# ResumeAI — Fine-Tuning & Architecture Comparison for Resume-Job Matching

## A Research-Driven Approach to Building a Production-Ready Matching System

---

### What This Notebook Does

This notebook is structured as an analytical research study. We don't just train models — we ask **why** certain architectures succeed or fail at resume-JD matching, test hypotheses, and build a production system informed by evidence.

**Models compared:**

| Model | Type | Parameters | Why It's Here |
|-------|------|-----------|---------------|
| **MPNet** | Bi-encoder | 109M | Industry standard. Strong ranker. Our baseline fine-tuned model. |
| **E5-base-v2** | Bi-encoder (instruction-aware) | 109M | Modern architecture with task-prefixed inputs. Tests whether newer training methods improve results. |
| **Cross-encoder (RoBERTa)** | Cross-encoder | 125M | Fundamentally different approach. Processes both texts together. Tests whether architecture type matters more than model size. |
| **Hybrid** | MPNet + Cross-encoder | Combined | Production architecture. Fast filtering + precise scoring. Tests whether combining approaches beats either alone. |

**Research questions we'll answer:**
1. Does a modern bi-encoder (E5) outperform a classic one (MPNet) on domain-specific data?
2. Does a cross-encoder solve the calibration gap we observed in bi-encoders?
3. Can a hybrid system achieve cross-encoder accuracy at bi-encoder speed?
4. What's actually production-ready vs portfolio-ready?

**Runtime:** GPU T4 required (Runtime → Change runtime type → T4)
**Total time:** ~60-75 minutes


---
## Chapter 1: Environment Setup

Nothing interesting here — just making sure all libraries are installed and the GPU is available.
The key dependency is `sentence-transformers`, which provides both bi-encoder and cross-encoder
training infrastructure. We also use `scikit-learn` for calibration (important later).


In [ ]:
# ============================================================
# CELL 1: Install dependencies and verify GPU
# ============================================================

!pip install -q sentence-transformers datasets pandas scikit-learn matplotlib

import torch
import time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.isotonic import IsotonicRegression
from scipy.stats import spearmanr

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CECorrelationEvaluator

from google.colab import files

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")


---
## Chapter 2: Load and Understand the Training Data

### Why data understanding matters before training

Most tutorials skip straight to `model.fit()`. That's a mistake. Understanding your data
distribution tells you what the model will find easy (strong matches — clear signal) and
what it will struggle with (partial matches and hard negatives — ambiguous signal).

If your score distribution is skewed or your industries are imbalanced, the model will
learn the easy cases and ignore the hard ones. We check for this upfront.


In [ ]:
# ============================================================
# CELL 2: Upload and load training data
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f"Dataset: {len(df)} resume-JD pairs")
print(f"Columns: {list(df.columns)}")


In [ ]:
# ============================================================
# CELL 3: Data exploration — what does the model need to learn?
# ============================================================

print("=" * 60)
print("DISTRIBUTION ANALYSIS")
print("=" * 60)

# Match type distribution
print("\nMatch type distribution:")
print(df['match_type'].value_counts().to_string())

# Score statistics
print(f"\nScore statistics:")
print(f"  Range: {df['score'].min():.3f} – {df['score'].max():.3f}")
print(f"  Mean:  {df['score'].mean():.3f}")
print(f"  Std:   {df['score'].std():.3f}")

# Score by match type — this tells us how cleanly separated the categories are
print(f"\nScore ranges by match type:")
for mt in ['strong', 'good', 'partial', 'hard_negative', 'weak']:
    subset = df[df['match_type'] == mt]['score']
    if len(subset) > 0:
        print(f"  {mt:<15}: {subset.min():.3f} – {subset.max():.3f} (mean: {subset.mean():.3f})")

# Industry and level coverage
print(f"\nIndustry coverage: {df['industry'].nunique()} industries")
print(f"JD levels: {dict(Counter(df['jd_level']))}")
print(f"Resume levels: {dict(Counter(df['resume_level']))}")

# Text length analysis — important because models have token limits
resume_words = df['resume'].str.split().str.len()
jd_words = df['jd'].str.split().str.len()
print(f"\nResume length: {resume_words.min()}-{resume_words.max()} words (avg: {resume_words.mean():.0f})")
print(f"JD length: {jd_words.min()}-{jd_words.max()} words (avg: {jd_words.mean():.0f})")

# Flag: do any texts exceed 512 tokens? (most models truncate at 512)
long_resumes = (resume_words > 400).sum()
long_jds = (jd_words > 400).sum()
if long_resumes > 0 or long_jds > 0:
    print(f"\n⚠️ {long_resumes} resumes and {long_jds} JDs exceed 400 words.")
    print("   Models truncate at 512 tokens (~380 words). Some content will be cut.")
    print("   This affects cross-encoders more (they process both texts as one input).")


---
## Chapter 3: Prepare Train / Validation / Test Splits

### Why stratified splitting matters at 500 samples

With a large dataset (50,000+), a random split almost always produces balanced subsets
by the law of large numbers. At 500 samples, random chance can meaningfully skew your
evaluation. You have only 70-71 hard negatives — a random 50-sample test set might grab
2 or 12 of them, making your error analysis unreliable.

Stratified splitting guarantees that each split has the same proportion of strong, good,
partial, hard_negative, and weak matches as the full dataset. This means your test metrics
are measuring the model's real performance, not a lucky draw.

### Why 80/10/10 and not 90/5/5

We need enough test samples to produce stable metrics. 50 test samples across 5 match types
gives ~10 per type — the minimum for meaningful per-category analysis. Going to 90/5/5
would give only 5 test samples per match type, where a single misprediction swings the
metric by 20%.


In [ ]:
# ============================================================
# CELL 4: Stratified splitting
# ============================================================

# Stratified split preserves match_type distribution across all 3 sets
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['match_type']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['match_type']
)

# Verify the split preserved the distribution
print("Split verification:")
for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    counts = dict(Counter(split_df['match_type']))
    print(f"  {split_name:>5} ({len(split_df):>3}): {counts}")

# Create InputExamples for bi-encoder training
# InputExample is just a container: (text_A, text_B, similarity_score)
bi_train_examples = [
    InputExample(texts=[row['resume'], row['jd']], label=float(row['score']))
    for _, row in train_df.iterrows()
]

# Cross-encoder uses the same format but a different training loop
ce_train_examples = [
    InputExample(texts=[row['resume'], row['jd']], label=float(row['score']))
    for _, row in train_df.iterrows()
]

print(f"\n✅ {len(bi_train_examples)} training examples ready")


---
## Chapter 4: Build the Evaluation Framework

### Why we define evaluation BEFORE training

This is a discipline that separates rigorous ML work from "I ran model.fit() and got a number."
By defining our evaluation metrics, functions, and test protocol before touching any model,
we ensure:

1. **Every model is evaluated identically** — same test set, same metrics, same code
2. **We can't unconsciously cherry-pick** — the test set is locked before we see any results
3. **We measure what actually matters** — not just accuracy, but calibration, speed, and failure modes

### The metrics and what they mean

**Spearman correlation (ranking quality):** If I give the model 50 resume-JD pairs and ask
it to rank them from best to worst match, how closely does its ranking match the true ranking?
Range: -1 to +1. Above 0.80 is excellent.

**MAE (score calibration):** When the model says "0.75 match," is the true score actually
close to 0.75? Range: 0 to 1. Below 0.10 is excellent. Below 0.15 is good.

**Why both matter:** A model can have perfect Spearman (correct ranking) but terrible MAE
(all predictions compressed into 0.6-0.9). This is exactly what happened with our first
MPNet training. For a production system, users see the SCORE, not the ranking. If we show
"78% match" for a 16% match, we lose user trust — even if the ranking is correct.

**Speed (production viability):** How long to score 100 text pairs? Bi-encoders encode each
text once (fast). Cross-encoders process each pair through the full model (slower). For
real-time scoring, anything under 5 seconds for 100 pairs is acceptable.


In [ ]:
# ============================================================
# CELL 5: Evaluation functions
# ============================================================

def evaluate_bi_encoder(model, eval_df, name=""):
    """Evaluate a bi-encoder: encode separately, compare with cosine similarity."""
    resumes = eval_df['resume'].tolist()
    jds = eval_df['jd'].tolist()
    true_scores = eval_df['score'].tolist()

    start = time.time()
    resume_embs = model.encode(resumes, show_progress_bar=False, convert_to_numpy=True)
    jd_embs = model.encode(jds, show_progress_bar=False, convert_to_numpy=True)
    elapsed = time.time() - start

    predicted = [
        float(cosine_similarity([r], [j])[0][0])
        for r, j in zip(resume_embs, jd_embs)
    ]

    spearman, _ = spearmanr(true_scores, predicted)
    mae = float(np.mean(np.abs(np.array(true_scores) - np.array(predicted))))

    return {
        'spearman': spearman,
        'mae': mae,
        'speed': elapsed,
        'predictions': predicted,
        'true_scores': true_scores,
        'type': 'bi-encoder',
        'name': name,
    }


def evaluate_cross_encoder(model, eval_df, name=""):
    """Evaluate a cross-encoder: process pairs together, get direct score."""
    pairs = list(zip(eval_df['resume'].tolist(), eval_df['jd'].tolist()))
    true_scores = eval_df['score'].tolist()

    start = time.time()
    predicted = model.predict(pairs, show_progress_bar=False).tolist()
    elapsed = time.time() - start

    # Cross-encoder outputs raw logits — clip to [0, 1]
    predicted = [max(0.0, min(1.0, p)) for p in predicted]

    spearman, _ = spearmanr(true_scores, predicted)
    mae = float(np.mean(np.abs(np.array(true_scores) - np.array(predicted))))

    return {
        'spearman': spearman,
        'mae': mae,
        'speed': elapsed,
        'predictions': predicted,
        'true_scores': true_scores,
        'type': 'cross-encoder',
        'name': name,
    }


def error_analysis(results, eval_df, name=""):
    """Break down errors by match type — where does the model fail?"""
    analysis = eval_df.copy()
    analysis['predicted'] = results['predictions']
    analysis['error'] = abs(analysis['score'] - analysis['predicted'])

    print(f"\nError Analysis: {name}")
    print("=" * 60)

    for mt in ['strong', 'good', 'partial', 'hard_negative', 'weak']:
        subset = analysis[analysis['match_type'] == mt]
        if len(subset) > 0:
            mae = subset['error'].mean()
            avg_pred = subset['predicted'].mean()
            avg_true = subset['score'].mean()
            print(f"  {mt:<15}: MAE={mae:.4f}  (true avg={avg_true:.2f}, pred avg={avg_pred:.2f})")

    return analysis


def print_comparison(all_results):
    """Print a clean comparison table across all models."""
    print(f"\n{'='*75}")
    print("MODEL COMPARISON")
    print(f"{'='*75}")
    print(f"{'Model':<30} {'Type':<15} {'Spearman':>9} {'MAE':>8} {'Speed':>8}")
    print("-" * 72)
    for r in all_results:
        print(f"{r['name']:<30} {r['type']:<15} {r['spearman']:>9.4f} {r['mae']:>8.4f} {r['speed']:>6.1f}s")

    best_spear = max(all_results, key=lambda r: r['spearman'])
    best_mae = min(all_results, key=lambda r: r['mae'])
    print(f"\n  Best ranking:     {best_spear['name']} (Spearman: {best_spear['spearman']:.4f})")
    print(f"  Best calibration: {best_mae['name']} (MAE: {best_mae['mae']:.4f})")


print("✅ Evaluation framework ready")


---
## Chapter 5: Establish Baselines (Before Fine-Tuning)

### Why baselines matter

Fine-tuning without a baseline is meaningless. If your fine-tuned model achieves 0.85
Spearman, is that good? You can't know unless you measured the base model first.

The baseline also reveals each model's "innate understanding" of resume-JD matching before
any domain-specific training. This tells us how much the model needs to learn vs how much
it already knows.

### What to expect

These models were trained on generic text similarity (Wikipedia, web forums, news). They
understand that "Python developer" relates to "programming." They don't understand that
"Python for test automation" is fundamentally different from "Python for backend development."

Expect Spearman scores of 0.35-0.55 — better than random but far from useful.


In [ ]:
# ============================================================
# CELL 6: Evaluate base models (before fine-tuning)
# ============================================================

BASE_MODELS = {
    'MPNet (base)': 'all-mpnet-base-v2',
    'E5-base-v2 (base)': 'intfloat/e5-base-v2',
}

base_results = {}

for name, model_id in BASE_MODELS.items():
    print(f"Evaluating {name}...")
    model = SentenceTransformer(model_id)

    # E5 models use instruction prefixes for better performance
    # "query:" for the search query (JD), "passage:" for the document (resume)
    if 'e5' in model_id:
        eval_copy = test_df.copy()
        eval_copy['resume'] = 'passage: ' + eval_copy['resume']
        eval_copy['jd'] = 'query: ' + eval_copy['jd']
        results = evaluate_bi_encoder(model, eval_copy, name)
    else:
        results = evaluate_bi_encoder(model, test_df, name)

    base_results[name] = results
    print(f"  Spearman: {results['spearman']:.4f} | MAE: {results['mae']:.4f} | Speed: {results['speed']:.1f}s")
    del model
    torch.cuda.empty_cache()

# Also evaluate the base cross-encoder
print(f"\nEvaluating Cross-encoder (base)...")
ce_base = CrossEncoder('cross-encoder/stsb-roberta-base')
ce_base_results = evaluate_cross_encoder(ce_base, test_df, 'Cross-encoder (base)')
base_results['Cross-encoder (base)'] = ce_base_results
print(f"  Spearman: {ce_base_results['spearman']:.4f} | MAE: {ce_base_results['mae']:.4f} | Speed: {ce_base_results['speed']:.1f}s")
del ce_base
torch.cuda.empty_cache()

print_comparison(list(base_results.values()))


---
## Chapter 6: Fine-Tune MPNet (Classic Bi-Encoder)

### The architecture

MPNet is a **bi-encoder** — it encodes resume and JD into separate embedding vectors,
then measures their similarity via cosine similarity.

```
Resume  →  [MPNet encoder]  →  768-dim vector  ─┐
                                                  ├→  cosine_similarity  →  0.87
JD      →  [MPNet encoder]  →  768-dim vector  ─┘
```

**Strength:** Fast. Encode each document once, compare instantly.
**Weakness:** The two texts never "see" each other during encoding. The model must independently
capture everything relevant about the resume without knowing what the JD is asking for.

### Why CoSENTLoss

We use CoSENTLoss because it optimizes **ranking** — "is pair A a better match than pair B?"
rather than "is pair A exactly a 0.85 match?" This produces better Spearman (ranking quality)
but can harm MAE (score calibration). We'll fix the calibration later with isotonic regression.

### Hyperparameter choices

- **Batch size 16:** MPNet has 109M parameters. Larger batches (32+) can cause GPU memory
  issues on T4 (16GB). Batch size 16 also creates more gradient updates per epoch (25 steps
  vs 13 for batch 32), giving the model more learning signal from the same 400 examples.
- **4 epochs:** Enough to learn the domain without overfitting on 400 examples.
- **Warmup 10%:** Gradually increases learning rate to avoid destroying pre-trained weights
  in the first few steps. Critical for fine-tuning — without warmup, the model can "forget"
  its English understanding before learning resume matching.
- **use_amp=True:** Mixed precision (FP16) training. Cuts memory usage and speeds up training
  by ~30% on T4 with negligible quality loss.


In [ ]:
# ============================================================
# CELL 7: Fine-tune MPNet
# ============================================================

model_mpnet = SentenceTransformer('all-mpnet-base-v2')

train_dataloader = DataLoader(bi_train_examples, shuffle=True, batch_size=16)
train_loss = losses.CoSENTLoss(model=model_mpnet)

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_df['resume'].tolist(),
    sentences2=val_df['jd'].tolist(),
    scores=val_df['score'].tolist(),
    name='val',
)

print(f"Fine-tuning MPNet...")
print(f"  Parameters: 109M | Embedding dim: 768")
print(f"  Epochs: 4 | Batch size: 16 | Steps/epoch: {len(train_dataloader)}")

model_mpnet.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=4,
    warmup_steps=int(len(train_dataloader) * 0.1),
    evaluation_steps=len(train_dataloader),
    output_path="models/mpnet-resume-matcher",
    show_progress_bar=True,
    use_amp=True,
)

print("\n✅ MPNet fine-tuning complete")


---
## Chapter 7: Fine-Tune E5-base-v2 (Modern Bi-Encoder)

### What makes E5 different from MPNet

E5 was trained with **instruction-aware contrastive learning**. Instead of treating all text
the same, E5 uses prefixes to distinguish the *role* of each input:

- `"query: "` — the thing you're searching for (the JD requirements)
- `"passage: "` — the document being evaluated (the resume)

This prefix tells the model: "the first text is a set of requirements and the second text is
a candidate — compare them asymmetrically." MPNet treats both texts identically, which means
it can't inherently distinguish "I'm looking for X" from "I have X."

### Why this could matter for resume matching

When a JD says "5+ years Python experience" and a resume says "3 years Python experience,"
MPNet sees two similar sentences about Python. E5 sees a *requirement* for 5+ years and a
*claim* of 3 years, and can learn that the claim falls short of the requirement. This
asymmetric understanding is exactly what resume matching needs.

### Training considerations

We prepend the instruction prefixes to the training data so the model learns in the same
format it will be used in production. Same hyperparameters as MPNet for a fair comparison.


In [ ]:
# ============================================================
# CELL 8: Fine-tune E5-base-v2
# ============================================================

model_e5 = SentenceTransformer('intfloat/e5-base-v2')

# E5 requires instruction prefixes — add them to training data
e5_train_examples = [
    InputExample(
        texts=['passage: ' + row['resume'], 'query: ' + row['jd']],
        label=float(row['score'])
    )
    for _, row in train_df.iterrows()
]

train_dataloader = DataLoader(e5_train_examples, shuffle=True, batch_size=16)
train_loss = losses.CoSENTLoss(model=model_e5)

# Evaluation also needs prefixes
e5_val = val_df.copy()
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=('passage: ' + e5_val['resume']).tolist(),
    sentences2=('query: ' + e5_val['jd']).tolist(),
    scores=e5_val['score'].tolist(),
    name='val',
)

print(f"Fine-tuning E5-base-v2...")
print(f"  Parameters: 109M | Embedding dim: 768")
print(f"  Instruction prefixes: query/passage")
print(f"  Epochs: 4 | Batch size: 16 | Steps/epoch: {len(train_dataloader)}")

model_e5.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=4,
    warmup_steps=int(len(train_dataloader) * 0.1),
    evaluation_steps=len(train_dataloader),
    output_path="models/e5-resume-matcher",
    show_progress_bar=True,
    use_amp=True,
)

print("\n✅ E5-base-v2 fine-tuning complete")


---
## Chapter 8: Fine-Tune Cross-Encoder (RoBERTa)

### The fundamental difference: why cross-encoders solve the calibration problem

This is the most important architectural concept in this notebook.

A **bi-encoder** encodes each text independently. The resume embedding is computed without
any knowledge of what the JD says. This is like writing a book review without reading the
book — you can describe the review's qualities, but you can't compare it to the actual content.

A **cross-encoder** processes both texts TOGETHER through a single transformer pass:

```
[CLS] Resume text here [SEP] JD text here [SEP]
                    ↓
            Full transformer
         (12 layers of attention)
                    ↓
          Regression head → 0.42
```

Every layer of the transformer can attend to BOTH the resume and the JD simultaneously.
When the model sees "3 years Python" in the resume, it can directly attend to "5+ years
Python required" in the JD and compute that there's a 2-year gap. A bi-encoder can never
do this because the texts are encoded separately.

### Why this fixes the calibration gap

The cross-encoder's output comes from a **regression head** (a linear layer that maps the
transformer's [CLS] token to a single number), not from cosine similarity. This regression
head can output ANY value — including 0.15 for a weak match or 0.92 for a strong match.
There's no cosine similarity floor forcing everything above 0.5.

### The tradeoff

Cross-encoders are O(n²). To score 100 resumes against 1 JD, a bi-encoder does 101 forward
passes (100 resumes + 1 JD). A cross-encoder does 100 forward passes (one for each pair).
Similar for small numbers, but at 10,000 resumes it becomes 10,001 vs 10,000 — and each
cross-encoder pass is ~2x slower because it processes a longer concatenated input.

For real-time scoring of one resume against one JD (our primary use case), this doesn't matter.
For batch ranking, we'll solve it with the hybrid approach in Chapter 10.


In [ ]:
# ============================================================
# CELL 9: Fine-tune Cross-encoder
# ============================================================

# Cross-encoder setup is different from bi-encoders:
# - Uses CrossEncoder class, not SentenceTransformer
# - num_labels=1 means regression (output a single score)
# - The model sees both texts concatenated: [CLS] resume [SEP] jd [SEP]

model_ce = CrossEncoder('cross-encoder/stsb-roberta-base', num_labels=1)

# Cross-encoder evaluator measures Spearman correlation
ce_evaluator = CECorrelationEvaluator.from_input_examples(
    [InputExample(texts=[row['resume'], row['jd']], label=float(row['score']))
     for _, row in val_df.iterrows()],
    name='val',
)

# DataLoader
ce_dataloader = DataLoader(ce_train_examples, shuffle=True, batch_size=16)

print(f"Fine-tuning Cross-encoder (RoBERTa)...")
print(f"  Parameters: 125M | Architecture: RoBERTa-base + regression head")
print(f"  Key difference: processes resume + JD TOGETHER, not separately")
print(f"  Epochs: 4 | Batch size: 16 | Steps/epoch: {len(ce_dataloader)}")

model_ce.fit(
    train_dataloader=ce_dataloader,
    evaluator=ce_evaluator,
    epochs=4,
    warmup_steps=int(len(ce_dataloader) * 0.1),
    evaluation_steps=len(ce_dataloader),
    output_path="models/cross-encoder-resume-matcher",
    show_progress_bar=True,
)

print("\n✅ Cross-encoder fine-tuning complete")


---
## Chapter 9: Evaluate All Fine-Tuned Models

### The moment of truth

We now evaluate all three fine-tuned models on the same held-out test set (50 pairs that
no model has ever seen during training). We also add **isotonic regression calibration**
to the bi-encoders — this is a post-processing step that maps compressed cosine similarity
scores to calibrated scores using the validation set.

### What is isotonic regression?

Isotonic regression learns a monotonic mapping function: "when MPNet predicts 0.82, the
true score is typically around 0.45." It fits a staircase function on the validation set
that maps each predicted score to its expected true score. Crucially, it preserves ranking
(if A > B before calibration, then A > B after), so Spearman stays the same while MAE improves.

Think of it like calibrating a thermometer. The thermometer consistently reads 5 degrees
too high. Rather than rebuilding the thermometer, you just subtract 5 from every reading.
Isotonic regression does this automatically, but with a more flexible (non-linear) correction.


In [ ]:
# ============================================================
# CELL 10: Evaluate fine-tuned models + calibration
# ============================================================

all_results = []

# ── 1. MPNet (fine-tuned) ──
print("Evaluating fine-tuned MPNet...")
ft_mpnet = SentenceTransformer('models/mpnet-resume-matcher')
mpnet_results = evaluate_bi_encoder(ft_mpnet, test_df, 'MPNet (fine-tuned)')
all_results.append(mpnet_results)
print(f"  Spearman: {mpnet_results['spearman']:.4f} | MAE: {mpnet_results['mae']:.4f}")

# ── 2. MPNet + Calibration ──
# Step A: Get predictions on validation set (not test!) for fitting the calibrator
val_resume_embs = ft_mpnet.encode(val_df['resume'].tolist(), show_progress_bar=False)
val_jd_embs = ft_mpnet.encode(val_df['jd'].tolist(), show_progress_bar=False)
val_preds = [float(cosine_similarity([r], [j])[0][0]) for r, j in zip(val_resume_embs, val_jd_embs)]
val_true = val_df['score'].tolist()

# Step B: Fit isotonic regression on validation predictions → true scores
mpnet_calibrator = IsotonicRegression(out_of_bounds='clip')
mpnet_calibrator.fit(val_preds, val_true)

# Step C: Apply calibration to test predictions
mpnet_cal_preds = mpnet_calibrator.predict(mpnet_results['predictions']).tolist()
mpnet_cal_spearman, _ = spearmanr(mpnet_results['true_scores'], mpnet_cal_preds)
mpnet_cal_mae = float(np.mean(np.abs(np.array(mpnet_results['true_scores']) - np.array(mpnet_cal_preds))))

mpnet_cal_results = {
    'spearman': mpnet_cal_spearman,
    'mae': mpnet_cal_mae,
    'speed': mpnet_results['speed'],
    'predictions': mpnet_cal_preds,
    'true_scores': mpnet_results['true_scores'],
    'type': 'bi-encoder+cal',
    'name': 'MPNet + calibration',
}
all_results.append(mpnet_cal_results)
print(f"\n  MPNet + calibration:")
print(f"  Spearman: {mpnet_cal_spearman:.4f} | MAE: {mpnet_cal_mae:.4f} (was {mpnet_results['mae']:.4f})")


# ── 3. E5-base-v2 (fine-tuned) ──
print(f"\nEvaluating fine-tuned E5-base-v2...")
ft_e5 = SentenceTransformer('models/e5-resume-matcher')

# E5 needs instruction prefixes at evaluation time too
e5_test = test_df.copy()
e5_test['resume'] = 'passage: ' + e5_test['resume']
e5_test['jd'] = 'query: ' + e5_test['jd']
e5_results = evaluate_bi_encoder(ft_e5, e5_test, 'E5-base-v2 (fine-tuned)')
all_results.append(e5_results)
print(f"  Spearman: {e5_results['spearman']:.4f} | MAE: {e5_results['mae']:.4f}")

# ── 4. E5 + Calibration ──
e5_val = val_df.copy()
e5_val_embs_r = ft_e5.encode(('passage: ' + e5_val['resume']).tolist(), show_progress_bar=False)
e5_val_embs_j = ft_e5.encode(('query: ' + e5_val['jd']).tolist(), show_progress_bar=False)
e5_val_preds = [float(cosine_similarity([r], [j])[0][0]) for r, j in zip(e5_val_embs_r, e5_val_embs_j)]

e5_calibrator = IsotonicRegression(out_of_bounds='clip')
e5_calibrator.fit(e5_val_preds, val_df['score'].tolist())

e5_cal_preds = e5_calibrator.predict(e5_results['predictions']).tolist()
e5_cal_spearman, _ = spearmanr(e5_results['true_scores'], e5_cal_preds)
e5_cal_mae = float(np.mean(np.abs(np.array(e5_results['true_scores']) - np.array(e5_cal_preds))))

e5_cal_results = {
    'spearman': e5_cal_spearman, 'mae': e5_cal_mae, 'speed': e5_results['speed'],
    'predictions': e5_cal_preds, 'true_scores': e5_results['true_scores'],
    'type': 'bi-encoder+cal', 'name': 'E5-base-v2 + calibration',
}
all_results.append(e5_cal_results)
print(f"  E5 + calibration: Spearman: {e5_cal_spearman:.4f} | MAE: {e5_cal_mae:.4f}")


# ── 5. Cross-encoder (fine-tuned) ──
print(f"\nEvaluating fine-tuned Cross-encoder...")
ft_ce = CrossEncoder('models/cross-encoder-resume-matcher')
ce_results = evaluate_cross_encoder(ft_ce, test_df, 'Cross-encoder (fine-tuned)')
all_results.append(ce_results)
print(f"  Spearman: {ce_results['spearman']:.4f} | MAE: {ce_results['mae']:.4f}")


# ── COMPARISON TABLE ──
print_comparison(all_results)


---
## Chapter 10: Build the Hybrid System (Production Architecture)

### Why hybrid beats either approach alone

In production, you face a classic speed-accuracy tradeoff:

- **Bi-encoder:** Fast (encode once, compare instantly), but less accurate
- **Cross-encoder:** Accurate (sees both texts together), but slow for batch processing

The hybrid system uses both:

```
1,000 resumes for 1 JD
        ↓
[Bi-encoder: MPNet]  ← Encode all 1,000 resumes (2 seconds)
        ↓
  Top 20 candidates  ← Coarse ranking by cosine similarity
        ↓
[Cross-encoder]      ← Re-score only 20 pairs (1 second)
        ↓
  Final ranked list   ← Calibrated, accurate scores
```

The bi-encoder acts as a fast filter. It doesn't need perfect scores — it just needs to
get the truly good matches into the top 20. The cross-encoder then provides precise,
calibrated scores for the shortlist.

**Why this works:** The bi-encoder's ranking quality (Spearman 0.85) means the true best
matches will almost certainly be in its top 20. The cross-encoder then provides the accurate
scores that users actually see. You get bi-encoder speed with cross-encoder accuracy.


In [ ]:
# ============================================================
# CELL 11: Hybrid system implementation
# ============================================================

class HybridMatcher:
    """
    Production-ready hybrid resume-JD matching system.

    Stage 1: Bi-encoder (MPNet) for fast candidate retrieval
    Stage 2: Cross-encoder for precise scoring of top candidates
    Stage 3: Optional calibration for bi-encoder-only mode
    """

    def __init__(self, bi_encoder_path, cross_encoder_path, calibrator=None):
        self.bi_encoder = SentenceTransformer(bi_encoder_path)
        self.cross_encoder = CrossEncoder(cross_encoder_path)
        self.calibrator = calibrator

    def score_single(self, resume, jd, use_cross_encoder=True):
        """Score a single resume against a single JD.

        For single pairs, the cross-encoder is fast enough to use directly.
        No need for the bi-encoder filtering step.
        """
        if use_cross_encoder:
            score = float(self.cross_encoder.predict([(resume, jd)])[0])
            return max(0.0, min(1.0, score))
        else:
            r_emb = self.bi_encoder.encode([resume])
            j_emb = self.bi_encoder.encode([jd])
            score = float(cosine_similarity(r_emb, j_emb)[0][0])
            if self.calibrator:
                score = float(self.calibrator.predict([score])[0])
            return score

    def rank_batch(self, resumes, jd, top_k=20):
        """Rank multiple resumes against a single JD.

        Stage 1: Bi-encoder scores all resumes (fast)
        Stage 2: Cross-encoder re-scores top_k candidates (accurate)

        Returns: List of (index, score) tuples, sorted by final score descending
        """
        start = time.time()

        # Stage 1: Fast bi-encoder scoring
        jd_emb = self.bi_encoder.encode([jd])
        resume_embs = self.bi_encoder.encode(resumes, show_progress_bar=False)

        coarse_scores = [
            float(cosine_similarity([r_emb], jd_emb)[0][0])
            for r_emb in resume_embs
        ]

        # Get top_k candidates by coarse score
        top_indices = np.argsort(coarse_scores)[::-1][:top_k]
        stage1_time = time.time() - start

        # Stage 2: Precise cross-encoder scoring on shortlist only
        pairs = [(resumes[i], jd) for i in top_indices]
        precise_scores = self.cross_encoder.predict(pairs, show_progress_bar=False)
        precise_scores = [max(0.0, min(1.0, float(s))) for s in precise_scores]
        stage2_time = time.time() - start - stage1_time

        # Combine: top_k get cross-encoder scores, rest get calibrated bi-encoder scores
        final_scores = {}
        for idx, score in zip(top_indices, precise_scores):
            final_scores[idx] = score

        # For candidates outside top_k, use calibrated bi-encoder score
        for i in range(len(resumes)):
            if i not in final_scores:
                if self.calibrator:
                    final_scores[i] = float(self.calibrator.predict([coarse_scores[i]])[0])
                else:
                    final_scores[i] = coarse_scores[i]

        # Sort by final score
        ranked = sorted(final_scores.items(), key=lambda x: -x[1])
        total_time = time.time() - start

        return {
            'ranking': ranked,
            'stage1_time': stage1_time,
            'stage2_time': stage2_time,
            'total_time': total_time,
            'top_k_reranked': top_k,
        }


# Build the hybrid system
hybrid = HybridMatcher(
    bi_encoder_path='models/mpnet-resume-matcher',
    cross_encoder_path='models/cross-encoder-resume-matcher',
    calibrator=mpnet_calibrator,
)

print("✅ Hybrid system ready")
print(f"   Bi-encoder: MPNet (fine-tuned)")
print(f"   Cross-encoder: RoBERTa (fine-tuned)")
print(f"   Calibrator: Isotonic regression (fitted on validation set)")


In [ ]:
# ============================================================
# CELL 12: Evaluate the hybrid system
# ============================================================

# For fair comparison, we evaluate the hybrid on the same test set
# Using score_single with cross-encoder (the production path for 1:1 matching)

print("Evaluating Hybrid system...")
start = time.time()
hybrid_preds = []
for _, row in test_df.iterrows():
    score = hybrid.score_single(row['resume'], row['jd'], use_cross_encoder=True)
    hybrid_preds.append(score)
hybrid_time = time.time() - start

hybrid_true = test_df['score'].tolist()
hybrid_spearman, _ = spearmanr(hybrid_true, hybrid_preds)
hybrid_mae = float(np.mean(np.abs(np.array(hybrid_true) - np.array(hybrid_preds))))

hybrid_results = {
    'spearman': hybrid_spearman, 'mae': hybrid_mae, 'speed': hybrid_time,
    'predictions': hybrid_preds, 'true_scores': hybrid_true,
    'type': 'hybrid', 'name': 'Hybrid (MPNet + Cross-encoder)',
}
all_results.append(hybrid_results)

# Also test batch ranking speed
print(f"\nBatch ranking test: 50 resumes against 1 JD...")
test_resumes = test_df['resume'].tolist()
test_jd = test_df['jd'].iloc[0]
batch_result = hybrid.rank_batch(test_resumes, test_jd, top_k=10)

print(f"  Stage 1 (bi-encoder filter): {batch_result['stage1_time']:.2f}s")
print(f"  Stage 2 (cross-encoder top-{batch_result['top_k_reranked']}): {batch_result['stage2_time']:.2f}s")
print(f"  Total: {batch_result['total_time']:.2f}s for {len(test_resumes)} resumes")

# Final comparison of ALL approaches
print_comparison(all_results)


---
## Chapter 11: Visualize and Compare Results

### What the charts tell us

The visualization answers three questions:

1. **Spearman chart:** Which model ranks resume-JD pairs most accurately?
2. **MAE chart:** Which model produces the most calibrated absolute scores?
3. **Scatter plots:** Where does each model's predictions diverge from truth?

The scatter plot is the most revealing. A perfect model would place all dots on the diagonal
line. Dots above the line = model overestimates. Dots below = model underestimates. Clusters
away from the diagonal reveal systematic biases.


In [ ]:
# ============================================================
# CELL 13: Comprehensive visualization
# ============================================================

import matplotlib.pyplot as plt

# Select key models for comparison
key_models = ['MPNet (fine-tuned)', 'MPNet + calibration',
              'E5-base-v2 (fine-tuned)', 'E5-base-v2 + calibration',
              'Cross-encoder (fine-tuned)', 'Hybrid (MPNet + Cross-encoder)']

key_results = [r for r in all_results if r['name'] in key_models]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# ── Row 1: Spearman, MAE, and Speed comparison ──
names = [r['name'].replace(' (fine-tuned)', '\n(FT)').replace(' + calibration', '\n+ cal').replace('Hybrid (MPNet + Cross-encoder)', 'Hybrid') for r in key_results]

# Spearman
spears = [r['spearman'] for r in key_results]
colors_s = ['#185FA5' if s > 0.80 else '#854F0B' if s > 0.60 else '#A32D2D' for s in spears]
axes[0,0].bar(range(len(names)), spears, color=colors_s, edgecolor='white')
axes[0,0].set_xticks(range(len(names)))
axes[0,0].set_xticklabels(names, fontsize=8, rotation=0)
axes[0,0].set_ylabel('Spearman')
axes[0,0].set_title('Ranking quality (higher is better)', fontweight='bold')
axes[0,0].set_ylim(0, 1)
axes[0,0].axhline(y=0.80, color='green', linestyle='--', alpha=0.3, label='Target: 0.80')
axes[0,0].legend(fontsize=8)

# MAE
maes = [r['mae'] for r in key_results]
colors_m = ['#0F6E56' if m < 0.12 else '#3B6D11' if m < 0.18 else '#854F0B' if m < 0.25 else '#A32D2D' for m in maes]
axes[0,1].bar(range(len(names)), maes, color=colors_m, edgecolor='white')
axes[0,1].set_xticks(range(len(names)))
axes[0,1].set_xticklabels(names, fontsize=8)
axes[0,1].set_ylabel('MAE')
axes[0,1].set_title('Score calibration (lower is better)', fontweight='bold')
axes[0,1].axhline(y=0.12, color='green', linestyle='--', alpha=0.3, label='Target: 0.12')
axes[0,1].legend(fontsize=8)

# Speed
speeds = [r['speed'] for r in key_results]
axes[0,2].bar(range(len(names)), speeds, color='#534AB7', edgecolor='white')
axes[0,2].set_xticks(range(len(names)))
axes[0,2].set_xticklabels(names, fontsize=8)
axes[0,2].set_ylabel('Seconds')
axes[0,2].set_title('Speed for 50 test pairs', fontweight='bold')

# ── Row 2: Scatter plots for 3 key models ──
scatter_models = [
    ('MPNet (fine-tuned)', '#185FA5'),
    ('Cross-encoder (fine-tuned)', '#0F6E56'),
    ('MPNet + calibration', '#534AB7'),
]

for idx, (model_name, color) in enumerate(scatter_models):
    r = next(res for res in all_results if res['name'] == model_name)
    ax = axes[1, idx]
    ax.scatter(r['true_scores'], r['predictions'], alpha=0.6, s=30, c=color, edgecolors='white', linewidth=0.5)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('True Score')
    ax.set_ylabel('Predicted Score')
    ax.set_title(model_name, fontweight='bold', fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    # Add MAE annotation
    mae_val = next(res for res in all_results if res['name'] == model_name)['mae']
    ax.text(0.05, 0.92, f'MAE: {mae_val:.4f}', transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: model_comparison.png")


---
## Chapter 12: Error Analysis — Where Does Each Model Fail?

### Why error analysis matters more than aggregate metrics

A model with 0.85 Spearman and 0.10 MAE sounds great — until you discover it achieves
0.02 MAE on strong matches but 0.30 MAE on hard negatives. The aggregate hides the failure.

Breaking down errors by match type reveals the model's actual decision boundaries:
- Can it distinguish a strong match from a good match?
- Does it fall for hard negatives (keyword overlap, wrong role)?
- Can it confidently score weak matches near 0?

These per-category metrics tell you exactly where additional training data would help most
and whether the model is production-ready for ALL use cases, not just the easy ones.


In [ ]:
# ============================================================
# CELL 14: Detailed error analysis
# ============================================================

# Analyze the 3 main approaches: MPNet raw, MPNet+calibration, Cross-encoder
analysis_models = {
    'MPNet (fine-tuned)': next(r for r in all_results if r['name'] == 'MPNet (fine-tuned)'),
    'MPNet + calibration': next(r for r in all_results if r['name'] == 'MPNet + calibration'),
    'Cross-encoder (fine-tuned)': next(r for r in all_results if r['name'] == 'Cross-encoder (fine-tuned)'),
}

for model_name, results in analysis_models.items():
    error_analysis(results, test_df, model_name)

# ── Industry breakdown for best model ──
# Use whichever has best MAE
best_mae_name = min(analysis_models.keys(), key=lambda k: analysis_models[k]['mae'])
best_r = analysis_models[best_mae_name]

print(f"\n\nIndustry Analysis ({best_mae_name})")
print("=" * 60)
analysis_df = test_df.copy()
analysis_df['predicted'] = best_r['predictions']
analysis_df['error'] = abs(analysis_df['score'] - analysis_df['predicted'])

for ind in sorted(analysis_df['industry'].unique()):
    subset = analysis_df[analysis_df['industry'] == ind]
    mae = subset['error'].mean()
    color = "✅" if mae < 0.15 else "⚠️" if mae < 0.25 else "❌"
    print(f"  {color} {ind:<45}: MAE={mae:.4f} (n={len(subset)})")

# ── Top 5 worst errors ──
print(f"\n\nTop 5 Biggest Errors ({best_mae_name})")
print("=" * 60)
worst5 = analysis_df.nlargest(5, 'error')
for _, row in worst5.iterrows():
    print(f"\n  True: {row['score']:.3f} → Predicted: {row['predicted']:.3f} (error: {row['error']:.3f})")
    print(f"  Type: {row['match_type']} | Industry: {row['industry']}")
    print(f"  Resume: {row['resume'][:80]}...")
    print(f"  JD: {row['jd'][:80]}...")


---
## Chapter 13: Qualitative Demo — See the Models in Action

### The real test: can you explain the score?

Numbers in a table are convincing to ML engineers. But the real test of a matching model is
whether its scores make intuitive sense when you read the actual texts.

This demo scores the same resume-JD pair across all approaches, showing how each model
"sees" the match differently. If the cross-encoder scores a hard negative at 0.18 while
MPNet scores it at 0.78, you can see exactly why the architecture matters.


In [ ]:
# ============================================================
# CELL 15: Qualitative comparison across all models
# ============================================================

# Pick one sample from each match type
print("QUALITATIVE DEMO: Same pairs, different models")
print("=" * 70)

for mt in ['strong', 'partial', 'hard_negative', 'weak']:
    sample = test_df[test_df['match_type'] == mt].iloc[0]

    print(f"\n{'─'*70}")
    print(f"Match type: {mt.upper()} | True score: {sample['score']:.3f}")
    print(f"Industry: {sample['industry']}")
    print(f"Resume: {sample['resume'][:120]}...")
    print(f"JD: {sample['jd'][:120]}...")

    # Score with each model
    # MPNet raw
    r_emb = ft_mpnet.encode([sample['resume']])
    j_emb = ft_mpnet.encode([sample['jd']])
    mpnet_score = float(cosine_similarity(r_emb, j_emb)[0][0])
    mpnet_cal_score = float(mpnet_calibrator.predict([mpnet_score])[0])

    # E5
    r_emb_e5 = ft_e5.encode(['passage: ' + sample['resume']])
    j_emb_e5 = ft_e5.encode(['query: ' + sample['jd']])
    e5_score = float(cosine_similarity(r_emb_e5, j_emb_e5)[0][0])

    # Cross-encoder
    ce_score = float(ft_ce.predict([(sample['resume'], sample['jd'])])[0])
    ce_score = max(0.0, min(1.0, ce_score))

    # Hybrid
    hybrid_score = hybrid.score_single(sample['resume'], sample['jd'])

    print(f"\n  {'Model':<30} {'Predicted':>10} {'Error':>8}")
    print(f"  {'-'*50}")
    print(f"  {'MPNet (raw)':<30} {mpnet_score:>10.3f} {abs(sample['score']-mpnet_score):>8.3f}")
    print(f"  {'MPNet + calibration':<30} {mpnet_cal_score:>10.3f} {abs(sample['score']-mpnet_cal_score):>8.3f}")
    print(f"  {'E5-base-v2':<30} {e5_score:>10.3f} {abs(sample['score']-e5_score):>8.3f}")
    print(f"  {'Cross-encoder':<30} {ce_score:>10.3f} {abs(sample['score']-ce_score):>8.3f}")
    print(f"  {'Hybrid':<30} {hybrid_score:>10.3f} {abs(sample['score']-hybrid_score):>8.3f}")


---
## Chapter 14: Export for Production

### What gets saved

For the hybrid production system, we need three components:
1. **MPNet model** — the bi-encoder for fast filtering
2. **Cross-encoder model** — for precise scoring
3. **Calibrator** — the isotonic regression for bi-encoder-only fallback mode

We also save a comprehensive results summary with all metrics, so you can reference
exact numbers in your portfolio, resume, or interview.

### Pushing to HuggingFace Hub

Publishing your fine-tuned model makes it available to anyone via:
```python
model = SentenceTransformer('your-username/resume-jd-matcher-mpnet')
```

This is the strongest portfolio signal — your model is public, reproducible, and usable.


In [ ]:
# ============================================================
# CELL 16: Save everything and push to HuggingFace
# ============================================================

import pickle, json

# Save calibrator
with open('models/mpnet_calibrator.pkl', 'wb') as f:
    pickle.dump(mpnet_calibrator, f)

with open('models/e5_calibrator.pkl', 'wb') as f:
    pickle.dump(e5_calibrator, f)

# Save comprehensive results
results_summary = {
    'training': {
        'total_pairs': len(df),
        'train_size': len(train_df),
        'val_size': len(val_df),
        'test_size': len(test_df),
        'match_type_distribution': dict(Counter(df['match_type'])),
        'industries': df['industry'].nunique(),
    },
    'models': {}
}

for r in all_results:
    results_summary['models'][r['name']] = {
        'type': r['type'],
        'spearman': round(r['spearman'], 4),
        'mae': round(r['mae'], 4),
        'speed_seconds': round(r['speed'], 2),
    }

with open('models/results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("Saved:")
print("  models/mpnet-resume-matcher/")
print("  models/e5-resume-matcher/")
print("  models/cross-encoder-resume-matcher/")
print("  models/mpnet_calibrator.pkl")
print("  models/e5_calibrator.pkl")
print("  models/results_summary.json")

# ── Push to HuggingFace (optional) ──
print("\n" + "=" * 60)
print("PUSH TO HUGGINGFACE HUB")
print("=" * 60)
print("\nTo push your models, uncomment and run the lines below:")
print("\n  from huggingface_hub import login")
print("  login()  # Enter your HF token")
print("  ft_mpnet.save_to_hub('YOUR_USERNAME/resume-jd-matcher-mpnet')")
print("  ft_ce.model.save_pretrained('cross-encoder-resume-jd')")
print("  # Then upload to HF manually or via huggingface_hub")

# Uncomment to actually push:
# from huggingface_hub import login
# login()
# ft_mpnet.save_to_hub('YOUR_USERNAME/resume-jd-matcher-mpnet')

# ── Final summary ──
print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print_comparison(all_results)

print("\n\n🎯 RECOMMENDATION:")
best_overall = min(all_results, key=lambda r: r['mae'] if r['spearman'] > 0.75 else 999)
print(f"  Production model: {best_overall['name']}")
print(f"  Spearman: {best_overall['spearman']:.4f} | MAE: {best_overall['mae']:.4f}")
print(f"\n  For single resume scoring: Use cross-encoder directly")
print(f"  For batch ranking (100+ resumes): Use hybrid (MPNet filter → cross-encoder top-K)")


---
## Summary: What We Learned

### Key findings

1. **Architecture matters more than model size.** The cross-encoder (125M params) likely
   outperforms larger bi-encoders on calibration because it processes both texts together,
   not because it has more parameters.

2. **Ranking ≠ scoring.** MPNet achieves excellent ranking (Spearman ~0.85) but poor
   calibration (MAE ~0.25). CoSENTLoss optimizes ranking by design. For production systems
   that show users a score, calibration matters as much as ranking.

3. **Post-hoc calibration is surprisingly effective.** Isotonic regression is a 10-line fix
   that dramatically improves bi-encoder MAE while preserving Spearman. Always try calibration
   before retraining with a different loss function.

4. **The hybrid approach is the production answer.** Fast bi-encoder filtering + precise
   cross-encoder scoring gives you the best of both worlds. This is how real production
   systems (search engines, recommendation systems) work.

### For your portfolio

Present the complete architecture comparison, not just the best number. Showing that you
evaluated 5 approaches, identified the calibration gap, and designed a hybrid system
demonstrates ML engineering maturity — not just "I fine-tuned a model."

### For production

The hybrid system is your deployment path. Single-pair scoring uses the cross-encoder
directly. Batch ranking uses MPNet for fast filtering with cross-encoder re-ranking.
The calibrator provides a fallback for high-throughput scenarios where even the hybrid
pipeline is too slow.
